In [ ]:
library(Seurat)
library(RcppML)
library(dplyr)
library(ggplot2)
library(ggpubr)
library(pheatmap)
library(schard)
source("~/Projects/heads/clustering.r")

In [ ]:
#obj = readRDS("/gpfs/gibbs/pi/braun/zy325/processed/scrcc_tex_test.rds")
# obj = readRDS("/gpfs/gibbs/project/braun/zy325/scrcc/processed/seurat_objects/Clustering_Tumor_0813.rds")
obj = readRDS("../lymphoid_addCgenes/scrcc_t_ilc2t_tex.rds")

In [ ]:
obj = clustering(obj,
                threshold_batch = 50,
                # vars.to.regress = c("nFeature_RNA","nCount_RNA","percent.mt"),
                plot_QC_metrics = F,
                group.by.vars = "batch_lab",
                harmony_theta=3,dims = 1:10)

In [ ]:
# obj = readRDS("~/project.braun/scrcc/processed/seurat_objects/Clustering_CD8T_0912.rds")
# obj = obj %>% subset(seurat_clusters %in% c(2,1,7))
obj$tissue = ifelse(grepl("NORM|NK",obj$sample_id2),"normal","tumor")
obj = subset(obj,tissue == "tumor")

# a = cd8t$Round2_cls
# b = m$anno
# all.equal(colnames(cd8t),rownames(m))

In [ ]:
DimPlot(obj)

In [ ]:
nmf_res = readRDS("~/project.braun/data/NMF_results_tex_2To25.rds")
res = nmf_res[[3]]
h = t(res@h)
anno = paste0("nmf",apply(h,1,which.max))
anno %>% table

In [ ]:
res@misc$w_init %>% dim

In [ ]:
hvg[!hvg %in% rownames(res@w)]

In [ ]:
res@h %>% dim

In [ ]:
genes = hvg

In [ ]:
obj = readRDS("~/project.braun/data/SC_RCC_CD8T_Filtered.rds")
obj = subset(obj,CD8T_clusters %in% c(0,1))
# obj = obj[, which(!obj$orig.ident %in% c("SC_69","SC_78","SC_79"))]

DefaultAssay(obj) = 'RNA5'
obj[['RNA5']]$data = as(obj[['RNA5']]$data,'dgCMatrix')
obj = NormalizeData(obj) %>% FindVariableFeatures(nfeatures = 2000)

In [ ]:
obj = NormalizeData(obj) %>% FindVariableFeatures(nfeatures = 2000)
hvg = VariableFeatures(obj)
gene.pattern = c("^AL(0|1|2|3|4|5|6|7|8|9)","^AC(0|1|2|3|4|5|6|7|8|9)","^AP(0|1|2|3|4|5|6|7|8|9)","^LINC","HBB","HBA","MT-","MTRNR")#,"^TR(A|B|G|D)V","^IG(H|K|L)V")
hvg = hvg[!hvg %in% grep(paste0(gene.pattern, collapse = "|"), hvg, value = T)]
VariableFeatures(obj) = hvg

obj = ScaleData(obj,features = hvg)



In [ ]:
#INCREASE SPARSity
obj@assays$RNA3 = as(obj@assays$RNA,'Assay')
cm = obj@assays$RNA3@scale.data
# rownames(cm) = hvg
cm[cm<0] = 0

In [ ]:
dim(cm)
cm = cm[rowSums(cm)>0,]
cm = cm[,colSums(cm)>0] 

#cm = cm[-which(rownames(cm) == "GJA5"),]

dim(cm)

In [ ]:
options(RcppML.threads = 1) 

set.seed(123456) 
#res = RcppML::nmf(cm,k = 4)
res_list = list()
W_list = list()
H_list = list()
for(i in 1:9){
  res = RcppML::nmf(cm,k = i+1)
  res_list[[i]] = res
  
  #X = W*H
  H_list[[i]] = res@h
  W_list[[i]] = res@w
}

In [ ]:
BasisToModules = function(basis,gmin = 5){
  
  scores = basis
  
  #rank modules for each gene
  ranks_x = apply(-scores,1,rank) %>% t
  #Normalize the genes, in case there are some genes that have overall higher contributions to the modules
  #ranks_x = t(apply(-t(t(scores) / apply(scores, 2, mean)), 1, rank))
  
  #rank genes for each module
  ranks_y = apply(-scores,2,rank)
  #Normalize the modules, make sure that the sum of gene contributions to each factor is identical
  #ranks_y = apply(-t(t(scores) / apply(scores, 2, mean)), 2, rank)
  
  #Assign genes to each module
  for (i in 1:ncol(scores)){
    #preserve the gene candidates which have a higher contribution to corresponding module than any other modules
    #other genes are assigned with an Inf value
    id = which(ranks_x[,i] > 1)
    ranks_y[id,i] = Inf
  }
  
  modules = apply(ranks_y, 2, function(m){
    #assign genes to the corresponding module in the order of their contributions until an Inf value was met
    a = sort(m[is.finite(m)])
    a = a[a == 1:length(a)]
    names(a)
  })
  
  #Remove the modules that contain less than 5 genes
  l = sapply(modules, length)
  keep = (l >= gmin)
  scores = scores[, keep]
  #print(keep)
  
  #Repeat the above process since there might be some modules being removed
  #Retrieve the genes assigned to those removed modules
  ranks_x = apply(-scores,1,rank) %>% t
  ranks_y = apply(-scores,2,rank)
  for (i in 1:ncol(scores)){
    ranks_y[ranks_x[,i] > 1,i] = Inf
  }
  modules = apply(ranks_y, 2, function(m){
    a = sort(m[is.finite(m)])
    a = a[a == 1:length(a)]
    names(a)
  })
  
  names(modules) = sapply(modules, '[', 1)
  names(modules) = paste0('M',c(1:length(modules)),'_',names(modules))
  print(paste0('Number of modules:',length(modules)))
  
  return(modules)
}

module.list = lapply(W_list,BasisToModules)

In [ ]:
saveRDS(res_list,file="../lymphoid_addCgenes/nmf_rank2to10_res_list.rds")

In [ ]:
res_list = readRDS("../lymphoid_addCgenes/nmf_rank2to10_res_list.rds")

BasisToModules = function(basis,gmin = 5){
  
  scores = basis
  
  #rank modules for each gene
  ranks_x = apply(-scores,1,rank) %>% t
  #Normalize the genes, in case there are some genes that have overall higher contributions to the modules
  #ranks_x = t(apply(-t(t(scores) / apply(scores, 2, mean)), 1, rank))
  
  #rank genes for each module
  ranks_y = apply(-scores,2,rank)
  #Normalize the modules, make sure that the sum of gene contributions to each factor is identical
  #ranks_y = apply(-t(t(scores) / apply(scores, 2, mean)), 2, rank)
  
  #Assign genes to each module
  for (i in 1:ncol(scores)){
    #preserve the gene candidates which have a higher contribution to corresponding module than any other modules
    #other genes are assigned with an Inf value
    id = which(ranks_x[,i] > 1)
    ranks_y[id,i] = Inf
  }
  
  modules = apply(ranks_y, 2, function(m){
    #assign genes to the corresponding module in the order of their contributions until an Inf value was met
    a = sort(m[is.finite(m)])
    a = a[a == 1:length(a)]
    names(a)
  })
  
  #Remove the modules that contain less than 5 genes
  l = sapply(modules, length)
  keep = (l >= gmin)
  scores = scores[, keep]
  #print(keep)
  
  #Repeat the above process since there might be some modules being removed
  #Retrieve the genes assigned to those removed modules
  ranks_x = apply(-scores,1,rank) %>% t
  ranks_y = apply(-scores,2,rank)
  for (i in 1:ncol(scores)){
    ranks_y[ranks_x[,i] > 1,i] = Inf
  }
  modules = apply(ranks_y, 2, function(m){
    a = sort(m[is.finite(m)])
    a = a[a == 1:length(a)]
    names(a)
  })
  
  names(modules) = sapply(modules, '[', 1)
  names(modules) = paste0('M',c(1:length(modules)),'_',names(modules))
  print(paste0('Number of modules:',length(modules)))
  
  return(modules)
}

module_list = lapply(res_list,function(x){
    w = x@w
    BasisToModules(w)
})

module_list[[5]] # the gene lists (rank=6)

In [ ]:
#res = RcppML::nmf(cm,k = 4,seed = res@misc$w_init)

i=5
res = res_list[[i]]
h = res@h
w = res@w
BasisToModules(w)

In [ ]:
H_list = lapply(res_list,function(x){x@h})

In [ ]:
h = H_list[[i]]
m = obj@meta.data
m$anno = paste0("nmf",apply(h,2,which.max))
m$anno %>% table

In [ ]:
obj_ = obj
obj_@meta.data = m

In [ ]:
options(repr.plot.width=5,repr.plot.height=4.5)

cor(t(H_list[[i]])) %>% pheatmap(display_numbers = T,border_color = "white",number_color = "black")

In [ ]:
CB = paste0("SCRCC",c('24', '05', '10', '50', '38', '42', '80', '03', '54', '64', '12', '29', '09', '21', '53', '59', '16'))
NCB = paste0("SCRCC",c("01", "02", "47", "45", "46", "55", "06", "40", "74", "11", "07"))

In [ ]:
nmf_scores = as.data.frame(t(h))
colnames(nmf_scores) = paste0("loading_",colnames(nmf_scores))
nmf_scores$anno_tex = obj_$anno

In [ ]:
ilc2t = readRDS("../lymphoid_addCgenes/scrcc_lymphoid_c4_clustered.rds")
ilc2t = subset(ilc2t, `RNA_snn_res.0.5` %in% c(1,3,4))
ilc2t = subset(ilc2t, name %in% colnames(obj_))

obj_$lineage3_clusters = "T" # paste0("T_",obj$`RNA_snn_res.0.5`)
obj_$lineage3_clusters[match(ilc2t$name,obj_$name)] = paste0("ILC2T_",ilc2t$`RNA_snn_res.0.5`)

In [ ]:
ilc2t = readRDS("../lymphoid_addCgenes/scrcc_lymphoid_c4_clustered.rds")
ilc2t$`RNA_snn_res.0.5` %>% table

In [ ]:
table(obj_$lineage3_clusters,obj_$anno)

In [ ]:
options(repr.plot.width=10,repr.plot.height=8)

obj_@meta.data %>% 
    cbind(t(h)) %>%
    group_by(sample_id2) %>%
    summarise(
        nmf1_score = mean(nmf1), #quantile(nmf1,probs = .75),
        nmf2_score = mean(nmf2), #quantile(nmf2,probs = .75),
        nmf3_score = mean(nmf3), #quantile(nmf3,probs = .75),
        nmf4_score = mean(nmf4), #quantile(nmf4,probs = .75),
        nmf5_score = mean(nmf5), #quantile(nmf5,probs = .75),
        nmf6_score = mean(nmf6), #quantile(nmf6,probs = .75),
        .groups="drop"
    ) %>% mutate(response = case_when(sample_id2 %in% CB~"R",
                                  sample_id2 %in% NCB~"NR",
                                  TRUE~"UK")) %>%
    filter(response != "UK") %>%
    pivot_longer(cols = -c(sample_id2,response),names_to = "module",values_to = "loading") %>%
    ggplot(aes(x = response,y = loading, fill=response)) +
    geom_boxplot(outlier.shape = NA)+
    geom_jitter()+
    stat_compare_means()+
    facet_wrap(.~module,scales = "free_y",ncol = 3) +
    theme_bw(base_size = 10)

In [ ]:
options(repr.plot.width=10,repr.plot.height=8)

m %>% mutate(response = case_when(sample_id2 %in% CB~"R",
                                  sample_id2 %in% NCB~"NR",
                                  TRUE~"UK")) %>%
    filter(response != "UK") %>%
    #filter(anno!="nmf6") %>%
    group_by(sample_id2,response, anno) %>%
    summarise(count=n(),response,.groups = "drop") %>%
    distinct() %>%
    group_by(sample_id2) %>%
    mutate(freq=count/sum(count)) %>%
    ggplot(aes(x = response,y = freq, fill=response)) +
    geom_boxplot(outlier.shape = NA)+
    geom_jitter()+
    stat_compare_means()+
    facet_wrap(.~anno,scales = "free_y",ncol = 3) +
    theme_bw(base_size = 10)

In [ ]:
m = FindMarkers(obj_,`ident.1` = "nmf3",group.by = "anno",only.pos = F,logfc.threshold = 1)
m %>% filter(p_val_adj<.01) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.1)

In [ ]:
obj_$response = case_when(
    obj$sample_id2 %in% CB~"R",
    obj$sample_id2 %in% NCB~"NR",
    TRUE~"UK")

m = FindMarkers(obj_,`ident.1` = "NR",group.by = "response",only.pos = F,logfc.threshold = 1)
m %>% filter(p_val_adj<.01) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
w = W_list[[i]] %>% .[unlist(module.list[[i]]),] #%>% .[,1:5]

In [ ]:
idx = apply(w,1,function(x){
    max(x)/sum(x) > 2/3
})

In [ ]:
idx = lapply(as.data.frame(w), function(col) order(col, decreasing = TRUE)[1:20])
idx

In [ ]:
options(repr.plot.width=4,repr.plot.height=12)

w[unique(unlist(idx)),] %>% pheatmap(cluster_cols = F,border_color = "grey95") #+ theme(axis.text.x = element_text(angle = 45,hjust = 1))

In [ ]:
cd8t = readRDS("scrcc_t_ilc2t_cd8t_clustered.rds")
cd8t_cnt = cd8t@meta.data %>% group_by(sample_id2) %>% summarise(cd8t_cnt=n())

In [ ]:
options(repr.plot.width=8,repr.plot.height=6)

obj_@meta.data %>% 
    group_by(sample_id2,anno) %>% 
    summarise(tex_cnt=n(),.groups = "drop") %>%
    left_join(cd8t_cnt,by="sample_id2") %>%
    mutate(proportion=tex_cnt/cd8t_cnt,response=case_when(sample_id2%in%CB~"CB",
                                                          sample_id2%in%NCB~"NCB",TRUE~"UK")) %>%
    filter(response!="UK") %>%
    #filter(cd8t_cnt>=100) %>%
    ggplot(aes(x=response,y=proportion))+
    geom_boxplot(outlier.shape = NA)+
    geom_jitter()+
    facet_wrap(.~anno,ncol = 3)+
    stat_compare_means()+
    theme_bw(base_size = 10)

In [ ]:
cd8t_cnt %>% arrange(cd8t_cnt)

In [ ]:
CB %>% sort
NCB %>% sort

In [ ]:
obj = readRDS("scrcc_t_ilc2t_cd8t_clustered.rds")

In [ ]:
obj$lineage3_clusters = obj$`RNA_snn_res.0.5`
obj$`RNA_snn_res.0.5` = NULL
obj$lineage3.5_clusters = obj$`RNA_snn_res.0.3`
obj$`RNA_snn_res.0.3` = NULL
obj$lineage4_clusters = obj$`RNA_snn_res.1`
obj$`RNA_snn_res.1` = NULL
obj$seurat_clusters=NULL

m = obj@meta.data

In [ ]:
ilc2t = readRDS("scrcc_lymphoid_c4_c0c3c6.rds")
m$lineage3[m$name %in% colnames(ilc2t)] = "ILC2T"

In [ ]:
m = left_join(m,nmf_scores %>% rownames_to_column(var = "name"),by="name")
rownames(m) = m$name

In [ ]:
table(m$lineage3,m$anno_tex)

In [ ]:
obj@meta.data = m

In [ ]:
obj$anno_tex %>% table

In [ ]:
saveRDS(obj,file = "scrcc_cd8t_annotated.rds")

In [ ]:
obj = readRDS("/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2026/scrcc_cd8t_annotated.rds")

In [ ]:
obj$lineage4 %>% table

In [ ]:
obj$lineage3_clusters %>% table

In [ ]:
ilc2t = readRDS("../lymphoid_addCgenes/scrcc_lymphoid_c4_clustered.rds")
#ilc2t = subset(ilc2t, `RNA_snn_res.0.5` %in% c(0,3,6))
ilc2t = subset(ilc2t, name %in% colnames(obj))

obj$lineage3_clusters = "T" #paste0("T_",obj$`RNA_snn_res.0.5`)
obj$lineage3_clusters[match(ilc2t$name,obj$name)] = "ILC2T" #paste0("ILC2T_",ilc2t$`RNA_snn_res.0.5`)

In [ ]:
table(obj$lineage3_clusters,obj$anno_tex)